In [1]:
import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
from sklearn.metrics import average_precision_score
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")

Train: (15624, 192)
Test:  (2000, 192)


In [3]:
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'verbosity': 0,
        'eval_metric': 'aucpr'
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    
    y_prob = model.predict_proba(X_test)[:, 1]
    auprc  = average_precision_score(y_test, y_prob)
    
    return auprc

print("Objective function định nghĩa xong!")

Objective function định nghĩa xong!


In [4]:
print("Bắt đầu tuning... (khoảng 3-5 phút)")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✅ Hoàn thành!")
print(f"Best AUPRC: {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Bắt đầu tuning... (khoảng 3-5 phút)


  0%|          | 0/50 [00:00<?, ?it/s]


✅ Hoàn thành!
Best AUPRC: 0.5813
Best params:
  n_estimators: 356
  max_depth: 4
  learning_rate: 0.29647270534084114
  subsample: 0.8370332874521356
  colsample_bytree: 0.8646330693461338
  min_child_weight: 1
  gamma: 0.015938381000568224


In [5]:
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score,
                             classification_report)

best_model = xgb.XGBClassifier(
    **study.best_params,
    random_state=42,
    verbosity=0,
    eval_metric='aucpr'
)
best_model.fit(X_train, y_train, verbose=False)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

f1    = f1_score(y_test, y_pred)
mcc   = matthews_corrcoef(y_test, y_pred)
auprc = average_precision_score(y_test, y_prob)
auc   = roc_auc_score(y_test, y_prob)

print(f"\n{'='*40}")
print(f"XGBoost TUNED vs DEFAULT")
print(f"{'='*40}")
print(f"{'Metric':<12} {'Default':>10} {'Tuned':>10} {'Cải thiện':>10}")
print(f"{'-'*42}")
print(f"{'F1':<12} {0.3579:>10.4f} {f1:>10.4f} {f1-0.3579:>+10.4f}")
print(f"{'AUPRC':<12} {0.5649:>10.4f} {auprc:>10.4f} {auprc-0.5649:>+10.4f}")
print(f"{'ROC-AUC':<12} {0.8812:>10.4f} {auc:>10.4f} {auc-0.8812:>+10.4f}")
print(f"{'MCC':<12} {0.4487:>10.4f} {mcc:>10.4f} {mcc-0.4487:>+10.4f}")

print(f"\n{classification_report(y_test, y_pred, target_names=['Legit','Fraud'])}")


XGBoost TUNED vs DEFAULT
Metric          Default      Tuned  Cải thiện
------------------------------------------
F1               0.3579     0.4554    +0.0975
AUPRC            0.5649     0.5813    +0.0164
ROC-AUC          0.8812     0.8918    +0.0106
MCC              0.4487     0.5269    +0.0782

              precision    recall  f1-score   support

       Legit       0.97      1.00      0.99      1923
       Fraud       0.96      0.30      0.46        77

    accuracy                           0.97      2000
   macro avg       0.97      0.65      0.72      2000
weighted avg       0.97      0.97      0.97      2000



In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

# Lưu model
best_model.save_model('../models/xgb_tuned.json')
print("✅ Đã lưu model!")

# Vẽ optimization history
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optimization History')

plt.subplot(1, 2, 2)
optuna.visualization.matplotlib.plot_param_importances(study)
plt.title('Hyperparameter Importance')

plt.tight_layout()
plt.savefig('../reports/optuna_results.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu biểu đồ!")

✅ Đã lưu model!
✅ Đã lưu biểu đồ!
